In [0]:
%pip install psycopg2

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import psycopg2

print(psycopg2.__version__)

2.9.13 (dt dec pq3 ext lo64)


In [0]:
from pyspark.sql import functions as F

# Conexão com o banco de dados PostgreSQL

PG_HOST = "pg-3117a07b-fixa-9a44.e.aivencloud.com"
PG_PORT = "15976"
PG_DATABASE = "db2anoPrincipal"
PG_SCHEMA = "public"

PG_USER = dbutils.secrets.get(
    scope="fixa",
    key="postgres-user"
)

PG_PASSWORD = dbutils.secrets.get(
    scope="fixa",
    key="postgres-password"
)

JDBC_URL = (
    f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
    "?sslmode=require"
)


In [0]:
def ler_postgres(nome_objeto):
    """
    Lê uma tabela ou view do PostgreSQL via JDBC.
    """
    objeto_completo = f"{PG_SCHEMA}.{nome_objeto}"

    return (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", objeto_completo)
        .option("user", PG_USER)
        .option("password", PG_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .option("fetchsize", "10000")
        .load()
    )

In [0]:
def adicionar_auditoria(df, nome_objeto):
    """
    Adiciona metadados técnicos da ingestão.
    """
    return (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_system",
            F.lit("postgresql")
        )
        .withColumn(
            "_source_object",
            F.lit(nome_objeto)
        )
    )

In [0]:
# Objetos necessários para as camadas Silver e Gold
OBJETOS = [
    "dim_categoria_equipamento",
    "dim_categoria_problema",
    "dim_instituicao",
    "dim_local",
    "dim_status_ordem_servico",
    "dim_status_problema",
    "dim_tecnico",
    "dim_tempo",
    "dim_unidade",
    "fato_ocorrencia",
    "fato_ordem_servico",
    "fato_problema",
    "tarefa"
]

In [0]:
# Carrega os objetos da origem e adiciona auditoria
bronze_dfs = {}

for objeto in OBJETOS:
    df = ler_postgres(objeto)

    bronze_dfs[objeto] = adicionar_auditoria(
        df,
        objeto
    )

In [0]:
# Validação simples da quantidade de registros carregados
for nome, df in bronze_dfs.items():
    print(
        f"{nome}: {len(df.take(1))} registros"
    )

dim_categoria_equipamento: 1 registros
dim_categoria_problema: 1 registros
dim_instituicao: 1 registros
dim_local: 1 registros
dim_status_ordem_servico: 1 registros
dim_status_problema: 1 registros
dim_tecnico: 1 registros
dim_tempo: 1 registros
dim_unidade: 1 registros
fato_ocorrencia: 1 registros
fato_ordem_servico: 1 registros
fato_problema: 1 registros
tarefa: 1 registros


In [0]:
# Conferência da fato principal de ordens de serviço
display(
    bronze_dfs["fato_ordem_servico"].limit(10)
)

ordem_servico_id,problema_id,tecnico_id,status_id,categoria_problema_id,unidade_id,local_id,data_criacao_id,data_prevista_id,data_inicio_id,data_conclusao_id,data_criacao,data_prevista,data_inicio,data_conclusao,prioridade,qtd_tarefas,qtd_reaberturas,qtd_reatribuicoes,tempo_pausado_minutos,tempo_espera_minutos,tempo_resolucao_minutos,esta_aberta,esta_atrasada,finalizada,foi_finalizada,concluida_no_prazo,foi_reaberta,foi_reatribuida,prioridade_nome,_ingestion_timestamp,_source_system,_source_object
41,41,40,3,1,7,41,20261021,20261102,20261021,null,2026-10-21T02:59:17.482Z,2026-11-02T02:59:17.482Z,2026-10-21T07:59:17.482Z,null,1,3,0,1,0E-18,300.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
42,42,40,3,2,7,42,20261020,20261101,20261020,null,2026-10-20T02:59:17.482Z,2026-11-01T02:59:17.482Z,2026-10-20T08:59:17.482Z,null,2,3,0,1,0E-18,360.000000000000000000,null,true,false,false,false,false,false,true,Baixa,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
43,43,46,3,3,8,43,20261019,20261031,20261019,null,2026-10-19T02:59:17.482Z,2026-10-31T02:59:17.482Z,2026-10-19T09:59:17.482Z,null,0,3,0,1,0E-18,420.000000000000000000,null,true,false,false,false,false,false,true,Alta,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
44,44,46,3,4,8,44,20261018,20261030,20261018,null,2026-10-18T02:59:17.482Z,2026-10-30T02:59:17.482Z,2026-10-18T10:59:17.482Z,null,1,3,0,1,0E-18,480.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
45,45,46,3,5,8,45,20261017,20261029,20261017,null,2026-10-17T02:59:17.482Z,2026-10-29T02:59:17.482Z,2026-10-17T11:59:17.482Z,null,2,3,0,1,0E-18,540.000000000000000000,null,true,false,false,false,false,false,true,Baixa,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
46,46,46,3,6,8,46,20261016,20261028,20261016,null,2026-10-16T02:59:17.482Z,2026-10-28T02:59:17.482Z,2026-10-16T12:59:17.482Z,null,0,3,0,1,0E-18,600.000000000000000000,null,true,false,false,false,false,false,true,Alta,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
47,47,46,3,7,8,47,20261015,20261027,20261015,null,2026-10-15T02:59:17.482Z,2026-10-27T02:59:17.482Z,2026-10-15T13:59:17.482Z,null,1,3,0,1,0E-18,660.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
48,48,46,3,8,8,48,20261014,20261026,20261014,null,2026-10-14T02:59:17.482Z,2026-10-26T02:59:17.482Z,2026-10-14T06:59:17.482Z,null,2,3,0,1,0E-18,240.000000000000000000,null,true,false,false,false,false,false,true,Baixa,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
49,49,52,3,9,9,49,20261013,20261025,20261013,null,2026-10-13T02:59:17.482Z,2026-10-25T02:59:17.482Z,2026-10-13T07:59:17.482Z,null,0,3,0,1,0E-18,300.000000000000000000,null,true,false,false,false,false,false,true,Alta,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico
50,50,52,3,10,9,50,20261012,20261024,20261012,null,2026-10-12T02:59:17.482Z,2026-10-24T02:59:17.482Z,2026-10-12T08:59:17.482Z,null,1,3,0,1,0E-18,360.000000000000000000,null,true,false,false,false,false,false,true,Média,2026-10-23T11:43:21.149Z,postgresql,fato_ordem_servico


In [0]:
# Conferência do schema recebido da fonte
bronze_dfs[
    "fato_ordem_servico"
].printSchema()

root
 |-- ordem_servico_id: long (nullable = true)
 |-- problema_id: long (nullable = true)
 |-- tecnico_id: long (nullable = true)
 |-- status_id: integer (nullable = true)
 |-- categoria_problema_id: integer (nullable = true)
 |-- unidade_id: long (nullable = true)
 |-- local_id: long (nullable = true)
 |-- data_criacao_id: integer (nullable = true)
 |-- data_prevista_id: integer (nullable = true)
 |-- data_inicio_id: integer (nullable = true)
 |-- data_conclusao_id: integer (nullable = true)
 |-- data_criacao: timestamp (nullable = true)
 |-- data_prevista: timestamp (nullable = true)
 |-- data_inicio: timestamp (nullable = true)
 |-- data_conclusao: timestamp (nullable = true)
 |-- prioridade: integer (nullable = true)
 |-- qtd_tarefas: integer (nullable = true)
 |-- qtd_reaberturas: integer (nullable = true)
 |-- qtd_reatribuicoes: integer (nullable = true)
 |-- tempo_pausado_minutos: decimal(38,18) (nullable = true)
 |-- tempo_espera_minutos: decimal(38,18) (nullable = true)
 |--

In [0]:
# Conferência da fato de solicitações
display(
    bronze_dfs["fato_problema"].limit(10)
)

problema_id,solicitante_id,unidade_id,local_id,categoria_equipamento_id,status_problema_id,data_criacao_id,data_criacao,quantidade_problema,esta_pendente,esta_aprovado,esta_reprovado,gerou_ordem_servico,_ingestion_timestamp,_source_system,_source_object
1,6,1,1,1,2,20260930,2026-09-30T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
2,5,1,2,2,2,20260923,2026-09-23T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
3,6,1,3,3,2,20260916,2026-09-16T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
4,5,1,4,4,2,20260909,2026-09-09T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
5,6,1,5,5,2,20260902,2026-09-02T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
6,5,1,6,6,2,20260826,2026-08-26T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
7,12,2,7,7,2,20260819,2026-08-19T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
8,11,2,8,8,2,20260812,2026-08-12T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
9,12,2,9,9,2,20260805,2026-08-05T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema
10,11,2,10,10,2,20260729,2026-07-29T02:59:17.482Z,1,false,true,false,true,2026-10-23T11:43:27.785Z,postgresql,fato_problema


In [0]:
# Conferência da fato de ocorrências
display(
    bronze_dfs["fato_ocorrencia"].limit(10)
)

ocorrencia_id,tecnico_id,unidade_id,local_id,categoria_problema_id,categoria_equipamento_id,data_criacao_id,data_criacao,prioridade,esta_ativo,quantidade_ocorrencia,prioridade_nome,_ingestion_timestamp,_source_system,_source_object
1,4,1,1,1,1,20261014,2026-10-14T02:59:17.482Z,0,true,1,Alta,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
2,3,1,2,2,2,20261011,2026-10-11T02:59:17.482Z,1,true,1,Média,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
3,4,1,3,3,3,20261008,2026-10-08T02:59:17.482Z,2,true,1,Baixa,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
4,3,1,4,4,4,20261005,2026-10-05T02:59:17.482Z,0,true,1,Alta,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
5,4,1,5,5,5,20261002,2026-10-02T02:59:17.482Z,1,true,1,Média,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
6,3,1,6,6,6,20260929,2026-09-29T02:59:17.482Z,2,true,1,Baixa,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
7,10,2,7,7,7,20260926,2026-09-26T02:59:17.482Z,0,true,1,Alta,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
8,9,2,8,8,8,20260923,2026-09-23T02:59:17.482Z,1,true,1,Média,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
9,10,2,9,9,9,20260920,2026-09-20T02:59:17.482Z,2,true,1,Baixa,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia
10,9,2,10,10,10,20260917,2026-09-17T02:59:17.482Z,0,true,1,Alta,2026-10-23T11:43:29.933Z,postgresql,fato_ocorrencia


In [0]:
# Conferência da tabela de tarefas usada no KPI OS-15
display(
    bronze_dfs["tarefa"].limit(10)
)

id,ordem_servico_id,status_ordem_servico,titulo,descricao,data_criacao,data_atualizacao,esta_ativo,_ingestion_timestamp,_source_system,_source_object
1,1,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-21T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
4,2,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-20T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
7,3,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-19T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
10,4,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-18T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
13,5,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-17T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
16,6,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-16T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
19,7,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-15T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
22,8,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-14T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
25,9,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-13T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa
28,10,0,Diagnosticar problema,Realizar diagnóstico inicial e identificar a causa do problema.,2026-10-12T03:59:17.482Z,2026-10-22T02:59:17.482Z,true,2026-10-23T11:43:32.029Z,postgresql,tarefa


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

DataFrame[]

In [0]:
# Persiste os dados brutos em Delta para consumo da Silver

for nome, df in bronze_dfs.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"workspace.bronze.{nome}"
        )
    )

    print(f"[OK] workspace.bronze.{nome}")

[OK] workspace.bronze.dim_categoria_equipamento
[OK] workspace.bronze.dim_categoria_problema
[OK] workspace.bronze.dim_instituicao
[OK] workspace.bronze.dim_local
[OK] workspace.bronze.dim_status_ordem_servico
[OK] workspace.bronze.dim_status_problema
[OK] workspace.bronze.dim_tecnico
[OK] workspace.bronze.dim_tempo
[OK] workspace.bronze.dim_unidade
[OK] workspace.bronze.fato_ocorrencia
[OK] workspace.bronze.fato_ordem_servico
[OK] workspace.bronze.fato_problema
[OK] workspace.bronze.tarefa
